# Query pilot server for pheontype data.
## What FHIR server to use?
Note this sample code is using a synthetic data server at: https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot/x1
The real server for URECA study(phs002921) is at: https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot/x1. You will first need to make Controlled Data Access Request(DAR). https://www.ncbi.nlm.nih.gov/projects/gap/cgi-bin/GetPdf.cgi?document_name=GeneralAAInstructions.pdf
## About the authorization token:
1. If you are using the synthetic server to try out FHIR pilot server, you do not need to have the "real token" in the token file. The script below still uses a token file so it works once you have the real token in the file.
2. If you want to use the real study data which is controlled-access, you need DAR approval. After your DAR is approved, go to https://www.ncbi.nlm.nih.gov/gap/power-user-portal/, login with your eRA account, scroll down and click on the "Task Specific Token" button to get the token file. Save the token in a text file. In the example below, it is saved to "task-specific-token-all.txt".
## What does this script do?
This sample script shows how to get the Study Subject Phenotype data.  You can see the content of the Subject Phenotype dataset here: https://www.ncbi.nlm.nih.gov/projects/gap/cgi-bin/dataset.cgi?study_id=phs002921.v2.p1&pht=12614 including the data dictionary (https://ftp.ncbi.nlm.nih.gov/dbgap/studies/phs002921/phs002921.v2.p1/pheno_variable_summaries/phs002921.v2.pht012614.v1.ICAC_Subject_Phenotypes.data_dict.xml ) 
Note that in dbGaP, the Subject Phenotype dataset usually includes demographic data in addition to phenotypic data.
## Script summary
This script first connects to the synthetic data server. Retrieves the patients of URECA study and saves it in a Python List: patient_ids. 
The script then iterates through the "patient_ids", to get the data in "subject phenotype" file which is stored in FHIR Observation Resource. 
The script saves the phenotype values in patient_observations.csv.  


In [ ]:
import os
import requests
import csv
from datetime import datetime
from time import sleep
from fhir_fetcher import fetch_all_data  # Ensure this module is available and handles paging through all records

def fetch_patient_observations(session, fhir_base_url, patient_id):
    qstr = f'Observation?subject=Patient/{patient_id}'
    # above qstr example:
    # https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot/x1/Observation?subject=Patient/4317770
    start_url = f"{fhir_base_url}/{qstr}"
    observations = fetch_all_data(session, start_url, 0)  # Fetch all observations for the patient
    return observations

def extract_observation_data(observations):
    data = {}
    for entry in observations:
        resource = entry.get('resource', {})
        code = resource.get('code', {}).get('coding', [{}])[0]
        attribute_name = code.get('display', '')
        value_string = resource.get('valueString', '')
        value_quantity = resource.get('valueQuantity', {}).get('value', '')
        if attribute_name:
            value = value_string if value_string else value_quantity
            if value:
                data[attribute_name] = value
    return data

def fetch_patient_ids(session, fhir_base_url, study_reference):
    
    
    query_url = f"{fhir_base_url}/ResearchSubject?study={study_reference}"
    # query_url example: https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot/x1/ResearchSubject?study=phs002921
    print ( query_url)
    research_subjects = fetch_all_data(session, query_url, 0, 'n')
    patient_ids = [entry['resource']['individual']['reference'].split('/')[-1] for entry in research_subjects]
    return patient_ids

def main():

    # 
    # If I had the patient_observations.csv open, then I am running this program, it will give an error when trying to write to it.
    # So check if the file is writable in the begining to avoid getting the error at the end of program after waiting for the program to finish:
    #
    output_file = 'patient_observations.csv'

    # Check if the file can be opened for writing
    try:
        with open(output_file, 'w', newline='') as csvfile:
            pass  # File opened successfully, nothing to write yet
    except PermissionError:
        print(f"Permission denied: Cannot open {output_file} for writing.")
        # Handle the error (e.g., exit the program or ask the user to close the file)
        exit(1)

# Rest of your program logic goes here

    
    starttime = datetime.now()
    starttimeStr = starttime.strftime('%Y-%m-%d %H:%M:%S')
    print("====== start time:", starttimeStr)

    ###############################################################################################
    # get the token from https://www.ncbi.nlm.nih.gov/gap/power-user-portal/.  
    #  Scroll down and click on the "Task Specific Token" button to get the light-weight version of the dbGaP RAS Passport.
    #  Save the file into a text. In my example, it is saved to .task-specific-token_all.txt. 
    ###############################################################################################
    TST_PATH = '~/dev/fhir/task-specific-token-all.txt'  
    fhir_base_url = "https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot/x1"
                   
    with open(os.path.expanduser(TST_PATH), 'r') as f:  
        tst_token = f.read().strip()
    
    session = requests.Session()
    session.headers.update({
        'Accept': 'application/fhir+json',
        'Authorization': f'Bearer {tst_token}',
        'Content-Type': 'application/x-www-form-urlencoded',
    })

    # study_reference = "phs002921.v2.p1.c1"
    study_reference = "phs002921"
 
    patient_ids = fetch_patient_ids(session, fhir_base_url, study_reference)
    print(f"Total patients fetched: {len(patient_ids)}")

    data = []
    columns = set()
    patients_with_observations = 0
    for patient_id in patient_ids:
        observations = fetch_patient_observations(session, fhir_base_url, patient_id)
        observation_data = extract_observation_data(observations)
        if observation_data:
            observation_data['Patient'] = patient_id
            columns.update(observation_data.keys())
            data.append(observation_data)
            patients_with_observations += 1
            # print(f"Observations obtained for patient: {patient_id}")
            print(f"Accumulative patients with observations: {patients_with_observations}")

        sleep(0.3)  # Add a delay of 1 second between each patient API request to avoid rate limits

    columns = ['Patient'] + sorted(columns)  # Ensure 'Patient' is the first column

    output_file = 'patient_observations.csv'
    with open(output_file, 'w', newline='') as csvfile:
        csvwriter = csv.DictWriter(csvfile, fieldnames=columns)
        csvwriter.writeheader()
        csvwriter.writerows(data)

    print(f"Data written to {output_file}")

    endtime = datetime.now()
    endtimeStr = endtime.strftime('%Y-%m-%d %H:%M:%S')
    print("====== end time:", endtimeStr)

    elapsed_time = endtime - starttime
    elapsed_seconds = elapsed_time.total_seconds()
    eminutes = elapsed_seconds // 60
    eseconds = elapsed_seconds % 60

    print(f"===========Elapsed time: {int(eminutes)} minutes and {int(eseconds)} seconds.")

if __name__ == "__main__":
    main()
